### FAISS (Facebook AI similarity search)

In [21]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 200, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [3]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln \nMemorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Winston Churchill (1940)On May 13, 1940, short

In [9]:
# ✅ NEW (recommended)
from langchain_ollama import OllamaEmbeddings
embedding = OllamaEmbeddings(model="nomic-embed-text")

db= FAISS.from_documents(docs, embedding)
db

In [14]:
query = "whom does speech address"

docs = db.similarity_search(query)

docs

[Document(id='6fade96e-65d5-4324-8a91-0f8a4c9e3461', metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln \nMemorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Win

## Retriever

In [15]:
retriver = db.as_retriever()
retriver.invoke(query)

[Document(id='6fade96e-65d5-4324-8a91-0f8a4c9e3461', metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln \nMemorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Win

In [16]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='6fade96e-65d5-4324-8a91-0f8a4c9e3461', metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln \nMemorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Wi

### CHROMADB

In [22]:
#Building sample vectorDB
from langchain_chroma import Chroma 
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [27]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln \nMemorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Winston Churchill (1940)On May 13, 1940, short

In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 0)
splits = text_splitter.split_documents(data)

In [29]:
from langchain_ollama import OllamaEmbeddings
embedding = OllamaEmbeddings(model="nomic-embed-text")

db= FAISS.from_documents(splits, embedding)
db

In [32]:
query = "who is the speaker?"
docs = db.similarity_search(query)
docs[0].page_content

'1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln'

In [ ]:
vectordb = Chroma.from_documents(documents=splits, embedding= embedding, persist_directory="./chroma_db")


In [35]:
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
docs = db2.similarity_search(query)
print(docs[0].page_content)

1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the 
steps of the Lincoln


In [36]:
retriver = vectordb.as_retriever()
retriver.invoke(query)[0].page_content

'1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln'